# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozair247/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.**  
This notebook audits my own Week‑5 model the way a careful reviewer would audit the
FlyRank research paper. It compares an honest client‑grouped split against a naive random
split, checks leakage, and rewrites my boldest claim in decision‑support language.

All data remains from `month=2026-03`. The sealed `month=2026-06` and `_sample` are not used.

In [1]:
%pip install -q duckdb scikit-learn pandas numpy
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

FACT_MONTH  = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

print("Connected.")

Connected.


Build the same feature frame as w05

In [2]:
feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions)                                                            AS impressions_month,
        SUM(f.gsc_clicks)                                                                  AS clicks_month,
        AVG(NULLIF(f.gsc_avg_position, 0))                                                 AS avg_position_month,
        SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0) * 100                        AS ctr_month,
        SUM(CASE WHEN f.ga4_data_available THEN f.ga4_engaged_sessions ELSE 0 END)         AS engaged_sessions_month,
        SUM(CASE WHEN f.ga4_data_available THEN f.ga4_sessions ELSE 0 END)                 AS sessions_month,
        MAX(CASE WHEN f.ga4_data_available THEN 1 ELSE 0 END)                              AS has_ga4_month,
        DATE_DIFF('day', ANY_VALUE(c.content_created_date), DATE '2026-03-31')             AS content_age_days,
        ANY_VALUE(c.word_count)                                                            AS word_count,
        SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impr_last15,
        SUM(CASE WHEN f.report_date <  DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impr_prev15
    FROM {FACT_MONTH} f
    LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    GROUP BY f.content_hash_id, f.client_hash_id
    HAVING SUM(f.gsc_impressions) > 0
""").df()

feature_frame = feature_frame.dropna(subset=["content_age_days"]).reset_index(drop=True)

feature_frame["is_declining_proxy"] = (
    (feature_frame["impr_prev15"] > 0)
    & ((feature_frame["impr_last15"] - feature_frame["impr_prev15"]) / feature_frame["impr_prev15"] < -0.20)
).astype(int)

feature_frame["engagement_rate_month"] = np.where(
    feature_frame["sessions_month"] > 0,
    feature_frame["engaged_sessions_month"] / feature_frame["sessions_month"] * 100,
    0.0,
)
feature_frame["has_word_count"] = feature_frame["word_count"].notna().astype(int)
feature_frame["word_count"] = feature_frame["word_count"].fillna(0)

print(f"Rows: {len(feature_frame)}")
print(f"Base rate: {feature_frame['is_declining_proxy'].mean():.3f}")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738
Base rate: 0.281


,content_hash_id,client_hash_id,impressions_month,clicks_month,avg_position_month,ctr_month,engaged_sessions_month,sessions_month,has_ga4_month,content_age_days,word_count,impr_last15,impr_prev15,is_declining_proxy,engagement_rate_month,has_word_count
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,0.0,5.331238,0.000000,0.0,0.0,0,47,2999,70.0,111.0,1,0.0,1
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,34.0,0.0,6.419872,0.000000,0.0,0.0,0,47,3281,14.0,20.0,1,0.0,1
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,77.0,0.0,4.888929,0.000000,0.0,0.0,0,47,3579,20.0,57.0,1,0.0,1
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,5.177774,0.000000,0.0,0.0,0,47,2993,83.0,246.0,1,0.0,1
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,602.0,4.0,4.428747,0.664452,0.0,0.0,0,47,2455,403.0,199.0,0,0.0,1


## 1. Two paper findings + my methodology questions

Below are two findings I noted from the FlyRank research paper, and the methodology
question I would ask about each. The goal is constructive scrutiny — the same way
I want my own work reviewed.

**Paper finding 1:**  
The paper reports that older pages with high impressions are more likely to be
flagged for a refresh.  

**My methodology question:**  
Where does the "needs refresh" label come from? Is it a direct editorial annotation,
a heuristic based on traffic drop, or a future outcome? If the label is derived from
the same features used to train the model (e.g., age and impressions), the reported
association may be circular. I would ask whether the label was defined independently
of the features.

**Paper finding 2:**  
The paper reports that low CTR among top‑ranking pages indicates a title/meta issue.  

**My methodology question:**  
Does the validation design support the claim that low CTR *causes* the page to need a
refresh? A cross‑sectional association between CTR and an outcome flag does not rule
out reverse causation (e.g., pages already in decline also have lower CTR). I would ask
whether the analysis used a time‑ordered design (CTR before the outcome) and whether
confounders like search intent were considered.

## 2. My model under an honest split (before/after)

In Week 5 I used a grouped split by `client_hash_id`, which is the honest choice.
Here I compare that against a naive random row split to show exactly how much
random splitting inflates the numbers. The model is the same Random Forest from
Week 5, using the same features and label.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score

FEATURES = [
    "content_age_days", "impressions_month", "ctr_month", "avg_position_month",
    "engagement_rate_month", "has_ga4_month", "word_count", "has_word_count",
]

X = feature_frame[FEATURES].copy()
y = feature_frame["is_declining_proxy"].copy()

# Handle NaNs
for col in ["avg_position_month", "ctr_month"]:
    X[col] = X[col].fillna(0)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

def train_and_evaluate(X_tr, y_tr, X_te, y_te):
    rf = RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=20,
        random_state=42, n_jobs=1
    )
    rf.fit(X_tr, y_tr)
    scores = rf.predict_proba(X_te)[:, 1]
    return {
        "precision@20": precision_at_k(scores, y_te.values, 20),
        "precision@50": precision_at_k(scores, y_te.values, 50),
        "auc": roc_auc_score(y_te, scores)
    }

# --- Before: random split (not honest) ---
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
before = train_and_evaluate(X_tr_rand, y_tr_rand, X_te_rand, y_te_rand)

# --- After: grouped by client (honest) ---
groups = feature_frame["client_hash_id"].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(gss.split(feature_frame, groups=groups))
X_tr_group, X_te_group = X.iloc[train_idx], X.iloc[test_idx]
y_tr_group, y_te_group = y.iloc[train_idx], y.iloc[test_idx]
after = train_and_evaluate(X_tr_group, y_tr_group, X_te_group, y_te_group)

comparison = pd.DataFrame({
    "Random split": before,
    "Grouped split": after
}).T
print(comparison.round(3))

               precision@20  precision@50    auc
Random split           0.70          0.66  0.698
Grouped split          0.55          0.56  0.649


**Observation:**  
The grouped split produces lower precision at both @20 and @50 compared with the random
split. This is expected: the random split allows the model to see content from the same
clients in both train and test, making the task easier. I keep the grouped split as the
honest number for any real decision‑support claim.

## 3. Leakage audit

The same check from Week 3, applied to the final feature set. The label's own ingredients
are `impr_last15` and `impr_prev15`; they must never appear as features. I also check for
FlyRank product flags and any future‑window columns.

In [4]:
future_window_cols = {"impr_last15", "impr_prev15"}
product_flag_cols = {"health_score", "priority_score", "action_type"}

feature_set = set(FEATURES)
overlap_label = feature_set & future_window_cols
overlap_flags = feature_set & product_flag_cols

print("Overlap between features and label ingredients (should be empty):", overlap_label)
print("Overlap between features and product flags (should be empty):", overlap_flags)
print()
print("Feature set:", feature_set)
print("Does any feature come from a future window? ", "No" if not overlap_label else "YES")

Overlap between features and label ingredients (should be empty): set()
Overlap between features and product flags (should be empty): set()

Feature set: {'word_count', 'avg_position_month', 'content_age_days', 'has_word_count', 'impressions_month', 'engagement_rate_month', 'ctr_month', 'has_ga4_month'}
Does any feature come from a future window?  No


In [5]:
# Error examples from the honest grouped split
rf = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20,
    random_state=42, n_jobs=1
)
rf.fit(X_tr_group, y_tr_group)
group_scores = rf.predict_proba(X_te_group)[:, 1]

test_scored = X_te_group.copy()
test_scored["model_score"] = group_scores
test_scored["true_label"] = y_te_group.values
test_scored["content_age_days"] = X_te_group["content_age_days"].values
test_scored["impressions_month"] = X_te_group["impressions_month"].values
test_scored["avg_position_month"] = X_te_group["avg_position_month"].values
test_scored["ctr_month"] = X_te_group["ctr_month"].values

top50 = test_scored.sort_values("model_score", ascending=False).head(50)
false_positives = top50[top50["true_label"] == 0]
print(f"False positives in top 50: {len(false_positives)} of 50")
print(false_positives[[
    "content_age_days", "impressions_month", "avg_position_month",
    "ctr_month", "model_score"
]].head(3))

False positives in top 50: 22 of 50
        content_age_days  impressions_month  avg_position_month  ctr_month  \
149147               228                1.0                42.0        0.0   
60561                228                1.0                 9.0        0.0   
149166               228                1.0                21.0        0.0   

        model_score  
149147     0.489157  
60561      0.484358  
149166     0.480930  


**Error examples:**  
The three highest‑confidence false positives are old pages with very low impressions
(e.g., age > 200 days, impressions = 1, CTR = 0). The proxy label cannot mark them as
declining because there is no prior baseline to decline from. This is a known blind spot:
the model confuses staleness with decline risk when traffic is near zero.

## 4. Claim rewrite

My boldest Week‑5 sentence was:

> “Random Forest beats the baseline and should be used to decide which pages to refresh.”

That claim overreaches. The evidence only shows a difference on one month, one label, and one
held‑out split. Here is the safe rewrite:

---

**Observed:** On a grouped client split of March 2026 data, Random Forest ranked pages with
precision@50 of **0.56**, compared with **0.04** for the Week‑4 baseline rule.  
**Measured:** The difference is 52 percentage points (0.56 − 0.04 = 0.52); the test set contained 15 clients not used in training.  
**Directional:** This suggests the model may be more useful than the fixed rule for ordering a
refresh review queue.  
**Decision‑support:** This result supports a pilot test of the model on the next evaluation
month, but it is not yet a production recommendation.

I will use this language in the capstone summary.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.